# BESOIN 4 : MODÈLE DE PRÉDICTION DE LA PUISSANCE NOMINALE

## Objectif
Développer un modèle d'intelligence artificielle capable de **prédire la catégorie de puissance nominale** d'une borne de recharge électrique (IRVE).

### Justification de l'approche par classification
La puissance nominale est une valeur continue, mais en pratique les bornes IRVE sont commercialisées selon des **paliers normalisés** (7.4 kW, 22 kW, 50 kW, etc.). On transforme donc ce problème en **classification multi-classes** selon les catégories officielles :
- **Lente** : ≤ 7.4 kW (bornes domestiques, prises EF/Type2 basiques)
- **Normale** : 7.4 – 22 kW (bornes AC résidentielles/parkings)
- **Accélérée** : 22 – 50 kW (bornes semi-rapides)
- **Rapide** : 50 – 150 kW (bornes DC rapides)
- **Ultra-rapide** : > 150 kW (stations haute puissance, CCS/CHAdeMO)

# 1. Importation des bibliothèques et configuration

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

import os
os.makedirs('fichier_pkl', exist_ok=True)
os.makedirs('figures', exist_ok=True)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

print('Bibliothèques chargées avec succès.')

# 2. Chargement et préparation des données

In [ ]:
df = pd.read_csv('data/export_IA.csv', low_memory=False)
print(f'Dimensions du dataset : {df.shape}')
print(f'\nColonnes disponibles :\n{list(df.columns)}')

## 2.1 Création de la variable cible

On discrétise la puissance nominale en 5 catégories selon les standards industriels IRVE.

In [ ]:
def categoriser_puissance(p):
    """Catégorise la puissance nominale selon les paliers standards IRVE."""
    if p <= 7.4:
        return 'Lente (<= 7.4 kW)'
    elif p <= 22:
        return 'Normale (7.4 - 22 kW)'
    elif p <= 50:
        return 'Acceleree (22 - 50 kW)'
    elif p <= 150:
        return 'Rapide (50 - 150 kW)'
    else:
        return 'Ultra-rapide (> 150 kW)'

TARGET = 'puissance_categorie'
df[TARGET] = df['puissance'].apply(categoriser_puissance)

ORDRE = [
    'Lente (<= 7.4 kW)',
    'Normale (7.4 - 22 kW)',
    'Acceleree (22 - 50 kW)',
    'Rapide (50 - 150 kW)',
    'Ultra-rapide (> 150 kW)'
]

print('Distribution de la cible :')
dist = df[TARGET].value_counts().reindex(ORDRE)
print(dist)
print(f'\nTotal : {dist.sum()} bornes')

## 2.2 Sélection et justification des features

### Justification du choix des variables

| Feature | Type | Justification |
|---|---|---|
| `implantation` | Catégorielle | Le type de lieu (station dédiée, parking, voirie) conditionne fortement la puissance installée |
| `nb_pdc` | Numérique | Les stations avec plus de points de charge tendent à avoir des puissances plus élevées |
| `prise_ccs` | Booléenne | La prise CCS est exclusivement utilisée pour la charge rapide DC (≥ 50 kW) |
| `prise_chademo` | Booléenne | CHAdeMO est aussi un standard DC rapide, corrélé aux fortes puissances |
| `prise_type2` | Booléenne | Type 2 couvre les charges lentes à accélérées (AC) |
| `prise_ef` | Booléenne | La prise EF (domestique) est limitée à ~3 kW, donc indicateur de charge lente |
| `gratuit` | Booléenne | Les bornes gratuites sont souvent des bornes lentes en voirie ou parkings publics |
| `condition_acces` | Catégorielle | Accès réservé/libre influence le type d'infrastructure déployée |

In [ ]:
FEATURES = ['implantation', 'nb_pdc', 'prise_ccs', 'prise_chademo',
            'prise_type2', 'prise_ef', 'gratuit', 'condition_acces']

data = df[FEATURES + [TARGET]].copy()
data = data.dropna(subset=[TARGET])
print(f'Shape après filtre cible : {data.shape}')

## 2.3 Nettoyage, Imputation et Encodage des données

In [ ]:
# --- Encodage booléens ---
bool_cols = ['prise_ccs', 'prise_chademo', 'prise_type2', 'prise_ef', 'gratuit']
for col in bool_cols:
    data[col] = data[col].map({
        True: 1, False: 0,
        'True': 1, 'False': 0,
        'TRUE': 1, 'FALSE': 0
    }).fillna(0).astype(int)

# --- Encodage implantation ---
data['implantation'] = data['implantation'].fillna('inconnu')
le_implantation = LabelEncoder()
data['implantation'] = le_implantation.fit_transform(data['implantation'])
joblib.dump(le_implantation, 'fichier_pkl/le_implantation_b4.pkl')
print(f'Classes implantation ({len(le_implantation.classes_)}) : {list(le_implantation.classes_)}')

# --- Encodage condition_acces ---
data['condition_acces'] = data['condition_acces'].str.strip().fillna('inconnu')
le_acces = LabelEncoder()
data['condition_acces'] = le_acces.fit_transform(data['condition_acces'])
joblib.dump(le_acces, 'fichier_pkl/le_acces_b4.pkl')
print(f'Classes condition_acces ({len(le_acces.classes_)}) : {list(le_acces.classes_)}')

# --- Imputation nb_pdc ---
mediane_pdc = data['nb_pdc'].median()
data['nb_pdc'] = data['nb_pdc'].fillna(mediane_pdc)
print(f'\nValeurs manquantes restantes :\n{data[FEATURES].isnull().sum()}')

# 3. Analyse Exploratoire des Données 

Cette section justifie le choix des variables via des visualisations.

In [ ]:
# --- Graphique 1 : Distribution des catégories de puissance ---
plt.figure(figsize=(12, 6))
counts = data[TARGET].value_counts().reindex(ORDRE)
bars = plt.bar(ORDRE, counts.values,
               color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6'])
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.title('Distribution des catégories de puissance nominale', fontsize=14, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylabel('Nombre de bornes')
plt.tight_layout()
plt.savefig('figures/distribution_puissance.png', dpi=300)
plt.show()
print('Observation : La classe "Normale" domine (déséquilibre de classes).')

In [ ]:
# --- Graphique 2 : Proportion des prises par catégorie (justification features) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Proportion des types de prises par catégorie de puissance\n(Justification du choix des features)',
             fontsize=13, fontweight='bold')

prises = ['prise_ccs', 'prise_chademo', 'prise_type2', 'prise_ef']
couleurs = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
titres = ['Prise CCS (DC Rapide)', 'Prise CHAdeMO (DC Rapide)',
          'Prise Type 2 (AC)', 'Prise EF (Domestique)']

temp = data.copy()

for ax, prise, couleur, titre in zip(axes.flatten(), prises, couleurs, titres):
    proportions = temp.groupby(TARGET)[prise].mean().reindex(ORDRE)
    ax.bar(range(len(ORDRE)), proportions.values, color=couleur, alpha=0.8)
    ax.set_title(titre, fontweight='bold')
    ax.set_xticks(range(len(ORDRE)))
    ax.set_xticklabels(['Lente', 'Normale', 'Accélérée', 'Rapide', 'Ultra-rapide'],
                       rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Proportion')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('figures/justification_features_prises_b4.png', dpi=300)
plt.show()
print('Observation : CCS et CHAdeMO sont fortement corrélés aux bornes rapides/ultra-rapides.')
print('La prise EF est quasi-exclusive aux bornes lentes. Ces patterns valident le choix des features.')

In [ ]:
# --- Graphique 3 : Heatmap de corrélation ---
plt.figure(figsize=(10, 7))
corr_data = data[FEATURES].copy()
corr_matrix = corr_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Matrice de corrélation des features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/correlation_features_b4.png', dpi=300)
plt.show()
print('Observation : Pas de forte multicolinéarité entre les features — bon signe pour les modèles linéaires.')

In [ ]:
# --- Graphique 4 : Distribution nb_pdc par catégorie ---
plt.figure(figsize=(12, 5))
data_plot = data[data['nb_pdc'] <= 20]
sns.boxplot(data=data_plot, x=TARGET, y='nb_pdc', order=ORDRE,
            palette='Set2', hue=TARGET, legend=False)
plt.title('Distribution du nb_pdc par catégorie de puissance', fontsize=13, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylabel('Nombre de points de charge')
plt.tight_layout()
plt.savefig('figures/boxplot_nbpdc_b4.png', dpi=300)
plt.show()
print('Observation : Les bornes ultra-rapides tendent à avoir plus de PDC par station.')

# 4. Préparation des données pour le Machine Learning

In [ ]:
X = data[FEATURES].astype(float)
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y  # important pour préserver la distribution des classes
)

print(f'Train : {X_train.shape[0]} échantillons | Test : {X_test.shape[0]} échantillons')
print(f'\nDistribution train :\n{y_train.value_counts()}')

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Sauvegarde du scaler et des features
joblib.dump(scaler, 'fichier_pkl/scaler_pretraitement_b4.pkl')
joblib.dump(list(FEATURES), 'fichier_pkl/features_b4.pkl')
print('\nScaler et features sauvegardés.')

# 5. Entraînement et comparaison de plusieurs algorithmes

## Justification du choix des algorithmes

Le cahier des charges demande de tester **plusieurs algorithmes** et de les comparer.

| Algorithme | Principe | Avantages pour ce problème |
|---|---|---|
| **Régression Logistique** | Modèle linéaire, probabilités via softmax | Baseline rapide, interprétable |
| **Random Forest** | Ensemble d'arbres de décision (bagging) | Robuste, gère bien les features mixtes, fournit l'importance des variables |
| **K-Nearest Neighbors** | Classification par les k voisins les plus proches | Non-paramétrique, bon pour des données avec des groupes distincts |
| **Gradient Boosting** | Ensemble d'arbres en série (boosting) | Souvent le plus performant sur données tabulaires |

In [ ]:
# Définition des modèles et de leurs grilles d'hyperparamètres
modeles_config = {
    'Régression Logistique': {
        'modele': LogisticRegression(random_state=RANDOM_STATE, max_iter=500),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'solver': ['lbfgs', 'saga']
        }
    },
    'Random Forest': {
        'modele': RandomForestClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5]
        }
    },
    'KNN': {
        'modele': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'Gradient Boosting': {
        'modele': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5]
        }
    }
}

print('Configuration des modèles prête.')
print(f'Nombre de modèles à comparer : {len(modeles_config)}')

In [ ]:
# GridSearchCV pour chaque modèle
resultats = {}

for nom, config in modeles_config.items():
    print(f'\n--- GridSearchCV : {nom} ---')
    
    grid = GridSearchCV(
        estimator=config['modele'],
        param_grid=config['params'],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train_scaled, y_train)
    
    meilleur = grid.best_estimator_
    y_pred = meilleur.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    resultats[nom] = {
        'modele': meilleur,
        'meilleurs_params': grid.best_params_,
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'y_pred': y_pred
    }
    
    print(f'  Meilleurs paramètres : {grid.best_params_}')
    print(f'  Accuracy : {acc:.4f} | F1-macro : {f1_macro:.4f} | F1-weighted : {f1_weighted:.4f}')

print('\n✓ GridSearchCV terminé pour tous les modèles.')

# 6. Comparaison et sélection du meilleur modèle

In [ ]:
# Tableau comparatif des performances
df_resultats = pd.DataFrame({
    'Algorithme': list(resultats.keys()),
    'Accuracy': [r['accuracy'] for r in resultats.values()],
    'F1-Score Macro': [r['f1_macro'] for r in resultats.values()],
    'F1-Score Weighted': [r['f1_weighted'] for r in resultats.values()]
}).sort_values('F1-Score Macro', ascending=False).reset_index(drop=True)

print('=' * 65)
print('       TABLEAU COMPARATIF DES PERFORMANCES')
print('=' * 65)
print(df_resultats.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 65)

meilleur_nom = df_resultats.iloc[0]['Algorithme']
print(f'\n→ Meilleur modèle sélectionné : {meilleur_nom}')

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Comparaison des performances des algorithmes', fontsize=14, fontweight='bold')

metriques = ['Accuracy', 'F1-Score Macro', 'F1-Score Weighted']
couleurs = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for ax, metrique in zip(axes, metriques):
    df_sorted = df_resultats.sort_values(metrique, ascending=True)
    bars = ax.barh(df_sorted['Algorithme'], df_sorted[metrique],
                   color=couleurs[:len(df_sorted)], alpha=0.85)
    for bar, val in zip(bars, df_sorted[metrique]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
    ax.set_title(metrique, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Score')

plt.tight_layout()
plt.savefig('figures/comparaison_algorithmes_b4.png', dpi=300)
plt.show()

# 7. Évaluation détaillée du meilleur modèle

In [ ]:
meilleur_modele = resultats[meilleur_nom]['modele']
y_pred_best = resultats[meilleur_nom]['y_pred']

print(f'RAPPORT D\'ÉVALUATION — {meilleur_nom}')
print('=' * 65)
print(classification_report(y_test, y_pred_best, target_names=ORDRE))

In [ ]:
# Matrice de confusion
matrice = confusion_matrix(y_test, y_pred_best, labels=ORDRE)
plt.figure(figsize=(10, 8))
sns.heatmap(
    matrice, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Lente', 'Normale', 'Accélérée', 'Rapide', 'Ultra-rapide'],
    yticklabels=['Lente', 'Normale', 'Accélérée', 'Rapide', 'Ultra-rapide']
)
plt.title(f'Matrice de Confusion — {meilleur_nom}', fontsize=13, fontweight='bold')
plt.ylabel('Réalité Terrain', fontsize=11)
plt.xlabel('Prédiction Modèle', fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('figures/matrice_confusion_b4.png', dpi=300)
plt.show()

In [ ]:
# Importance des features (disponible pour Random Forest et Gradient Boosting)
if hasattr(meilleur_modele, 'feature_importances_'):
    importances = meilleur_modele.feature_importances_
    df_imp = pd.DataFrame({
        'Feature': FEATURES,
        'Importance': importances
    }).sort_values('Importance', ascending=True)

    plt.figure(figsize=(10, 6))
    colors = ['#e74c3c' if imp > df_imp['Importance'].median() else '#3498db'
              for imp in df_imp['Importance']]
    bars = plt.barh(df_imp['Feature'], df_imp['Importance'], color=colors, alpha=0.85)
    for bar, val in zip(bars, df_imp['Importance']):
        plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)
    plt.title(f'Importance des Features — {meilleur_nom}',
              fontsize=13, fontweight='bold')
    plt.xlabel('Importance (Gini)')
    plt.axvline(df_imp['Importance'].median(), color='gray',
                linestyle='--', alpha=0.6, label='Médiane')
    plt.legend()
    plt.tight_layout()
    plt.savefig('figures/feature_importance_b4.png', dpi=300)
    plt.show()
    print('→ Les features les plus importantes confirment les choix faits en section 2.2.')
else:
    print(f'Importance des features non disponible pour {meilleur_nom}.')
    print('(Disponible pour Random Forest et Gradient Boosting)')

## 7.1 Discussion des résultats

### Analyse par classe
- **Normale (7.4–22 kW)** : classe majoritaire → le modèle prédit bien car elle domine les données (biais possible).
- **Rapide (50–150 kW)** et **Accélérée (22–50 kW)** : classes minoritaires → performances plus faibles, confusion entre les deux.
- **Ultra-rapide (>150 kW)** : bien identifiée grâce aux prises CCS/CHAdeMO comme indicateurs forts.
- **Lente (≤7.4 kW)** : bien identifiée grâce à la prise EF.

### Limites
- Le **déséquilibre de classes** (61k Normale vs 13k Rapide) pénalise les classes minoritaires.
- Des techniques comme **SMOTE** ou la pondération des classes pourraient améliorer les résultats.
- L'ajout de features géographiques (département, région) pourrait également aider.

In [ ]:
# F1-score par classe pour le meilleur modèle
from sklearn.metrics import f1_score as f1_per_class

f1_classes = f1_per_class(y_test, y_pred_best, average=None, labels=ORDRE)
df_f1 = pd.DataFrame({'Classe': ORDRE, 'F1-Score': f1_classes})

plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if f > 0.6 else '#e74c3c' for f in f1_classes]
bars = plt.bar(range(len(ORDRE)), f1_classes, color=colors, alpha=0.85)
for bar, val in zip(bars, f1_classes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
plt.axhline(0.6, color='gray', linestyle='--', alpha=0.6, label='Seuil 0.6')
plt.xticks(range(len(ORDRE)), ['Lente', 'Normale', 'Accélérée', 'Rapide', 'Ultra-rapide'],
           rotation=15, ha='right')
plt.ylabel('F1-Score')
plt.ylim(0, 1.1)
plt.title(f'F1-Score par classe — {meilleur_nom}', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('figures/f1_par_classe_b4.png', dpi=300)
plt.show()

print(df_f1.to_string(index=False))

# 8. Sauvegarde du modèle final

In [ ]:
joblib.dump(meilleur_modele, 'fichier_pkl/modele_classification_b4.pkl')

# Sauvegarde du nom du meilleur modèle pour référence
joblib.dump(meilleur_nom, 'fichier_pkl/nom_meilleur_modele_b4.pkl')

print('Fichiers sauvegardés :')
print('  - fichier_pkl/modele_classification_b4.pkl  (modèle entraîné)')
print('  - fichier_pkl/scaler_pretraitement_b4.pkl   (scaler StandardScaler)')
print('  - fichier_pkl/le_implantation_b4.pkl        (encodeur implantation)')
print('  - fichier_pkl/le_acces_b4.pkl               (encodeur condition_acces)')
print('  - fichier_pkl/features_b4.pkl               (liste des features)')
print(f'\nModèle final : {meilleur_nom}')
print(f'Accuracy test : {resultats[meilleur_nom]["accuracy"]:.4f}')
print(f'F1-Score macro : {resultats[meilleur_nom]["f1_macro"]:.4f}')

# 9. Récapitulatif

## Ce qui a été réalisé

1. **Préparation des données** : nettoyage, encodage (LabelEncoder), imputation (médiane), normalisation (StandardScaler). Tous les objets de prétraitement sont sauvegardés en `.pkl`.

2. **Analyse exploratoire** : visualisation de la distribution cible, corrélation features/cible (types de prises), heatmap de corrélation, boxplot nb_pdc.

3. **Comparaison de 4 algorithmes** via GridSearchCV (Régression Logistique, Random Forest, KNN, Gradient Boosting).

4. **Métriques** : Accuracy, F1-score macro, F1-score weighted, rapport de classification complet, matrice de confusion, F1 par classe.

5. **Sauvegarde** du meilleur modèle utilisable par le script `predict_puissance.py`.

## Conclusion
Le meilleur modèle obtenu est **{meilleur_nom}** avec une accuracy de **{:.2f}%** sur le jeu de test. Les classes extrêmes (Lente, Ultra-rapide) sont mieux prédites grâce aux indicateurs forts que sont les types de prises. Les classes intermédiaires (Accélérée, Rapide) restent plus difficiles à distinguer — une piste d'amélioration serait d'utiliser des techniques de rééquilibrage des classes.